[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/05_control_and_tracing.ipynb)


# Agentic Systems Foundations
## Notebook 05: Control and Tracing — Stopping, and Seeing
**Duration:** 25 min &nbsp;|&nbsp; **Mode:** Demonstration

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** the **[terminate?]** check, and the record of everything the loop did.


In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Makes `agent_core` importable whether you are in Colab, in a local venv,
# or running from a clone. Installs nothing you do not need: the package's
# only hard dependency is the Python standard library.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


if IN_COLAB:
    # openai for the real provider; jsonschema + langchain for the parallel
    # mappings shown in notebook 03. All optional — the notebook degrades
    # gracefully if any is missing.
    _pip("openai", "python-dotenv", "jsonschema", "langchain-core")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

# Where the Acme data lives — the tools resolve this automatically, but we
# print it so a path problem is visible immediately rather than as an empty
# search result three cells later.
from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDER  (works with NO key at all)
# ============================================================
# Default stack = OpenAI gpt-4o-mini with native tool calling.
# In Colab the key is read from the SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON -> re-run this cell.
#
# With NO key we fall back to MockToolCallLLM. Read that name literally:
# unlike a text-only mock, it DECIDES TOOL CALLS, so the entire agent loop —
# every notebook in this session — runs offline and deterministically.
import os

def _load_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

HAS_KEY = _load_key()
os.environ.setdefault("AGENT_LLM_PROVIDER", "openai" if HAS_KEY else "mock")

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
if not HAS_KEY:
    print("\nNo API key found -> running on the offline mock.")
    print("Everything in this notebook still works. Outputs are labelled [mock].")

## WHY — `while True:` is a complete agent and an irresponsible one

The gap between a demo agent and a deployable one is almost entirely here.

A loop that calls an LLM and a tool is a fifteen-minute exercise — you wrote one
in notebook 02. A loop that reliably **stops** — on success, on failure, on
running out of money, on realising it is going in circles — is the actual
engineering.

Three genuinely different questions live in this step, and conflating them is
the standard mistake:

1. **Are we done?** — the goal is satisfied. The good ending.
2. **Must we stop?** — the budget is gone. The safe ending.
3. **Are we stuck?** — budget remains, but nothing is progressing. **The
   interesting ending, and the one nobody implements until it has cost them.**

Question 3 is where the money actually goes. An agent repeating a failing call
has exceeded no limit; it is simply not going anywhere, and it will keep not
going anywhere until the step budget runs out.


## WHAT — four conditions, and why each exists

| Condition | Fires when | Catches what nothing else does |
|---|---|---|
| `budget_exceeded` | steps / calls / seconds hit a limit | the backstop. Never optional. |
| `repetition` | same call, **same arguments**, N times | the agent already has the answer and cannot tell |
| `error_streak` | N consecutive failures | flailing with *different* wrong arguments — invisible to `repetition` |
| `no_new_information` | last N *successful* calls returned identical results | varied arguments, all succeeding, learning nothing — invisible to both of the above |

`no_new_information` is the subtle one. Three differently-worded searches
returning the same passage: the call log looks healthy, every call is green, and
the *information* is flat. It is the condition that most closely tracks what a
human supervisor would notice, and the one most often missing.

### Order matters more than you would think

The **first** condition to fire is the one recorded, so put the most
*diagnostic* conditions first:

```python
TerminationPolicy([repetition(), error_streak(), no_new_information(),
                   context_limit(), budget_exceeded()])
```

Check `budget` first and every trace says `max_steps (8) reached` — technically
true, diagnostically useless. Same conditions, same behaviour, reordered:
completely different debugging experience.


> ### ✋ Predict before you run
> We are about to run one deliberately looping agent under two policies: the full diagnostic set, and a budget-only policy of the kind most first agents have. Both stop. **How many steps will each take, and — more importantly — what will each `stop_reason` tell you about what went wrong?**
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# Same broken agent, two termination policies.
from agent_core import TerminationPolicy, broken_agent, compare

goal = "What is the status of order ACME-1042?"

diagnostic  = broken_agent("no_progress_loop")
budget_only = broken_agent("no_progress_loop", policy=TerminationPolicy.budget_only())

t_diag   = diagnostic.run(goal).trace
t_budget = budget_only.run(goal).trace

print(compare({"diagnostic": t_diag, "budget-only": t_budget}))
print()
print("diagnostic  stop_reason:", t_diag.stop_reason)
print("budget-only stop_reason:", t_budget.stop_reason)

**What you should observe:** both stop, so both are "safe". They are not
equally useful.

- Budget-only burns the **whole** step budget and reports `max_steps reached`.
  That tells you the agent ran out. It tells you nothing about *why*.
- The diagnostic policy stops in **3 steps** and reports `repetition: identical
  call repeated 3× — get_order_status({"order_id": "ACME-1042"})`. That names the
  problem and hands you the fix.

Same safety. A fraction of the cost. And the difference between a bug you can
act on and a bug you have to reproduce first.


## HOW — the conditions, individually

In [ ]:
# Each condition, and the situation only it catches.
from agent_core import (TerminationPolicy, budget_exceeded, error_streak,
                        no_new_information, repetition)
from agent_core.control import context_limit

policy = TerminationPolicy()
print("default policy order:", " → ".join(policy.names()))
print()

for name, cond in [("repetition", repetition(threshold=3)),
                   ("error_streak", error_streak(limit=3)),
                   ("no_new_information", no_new_information(window=3)),
                   ("context_limit", context_limit(max_tokens=12000)),
                   ("budget", budget_exceeded())]:
    print(f"{name:<20} {cond.check.__doc__ or ''}".rstrip())

print("\nEach is a NAMED predicate that returns a REASON string, not a bool.")
print("That is why the trace can say what stopped it instead of just that it stopped.")

In [ ]:
# Budgets are per-run and copied, not shared. Watch a tight budget bite.
from agent_core import Agent, Budget

tight = Agent(budget=Budget(max_steps=2, max_tool_calls=2))
run = tight.run("Check the status of order ACME-1043 and whether I can refund it for changed_mind.")

print("status     :", run.state.status.value)
print("stop_reason:", run.state.stop_reason)
print("\nAnswer given despite stopping early:\n")
print(run.answer[:420])

Notice it still produced an answer. Stopping early is not an excuse for silence:
the user asked a question and deserves to know that we tried, what we found, and
that we stopped. **Partial evidence plus an honest "I did not finish" beats both
silence and a confident answer synthesised from nothing.**


## WHY — tracing is the debugging interface for the whole paradigm

A pipeline fails in one place and a stack trace points at it.

An agent fails **across time**: it chose a reasonable tool at step 2, got a
slightly wrong result, believed it at step 3, and produced a confidently
incorrect answer at step 5. Nothing threw. There is no line number.

So "why did the agent do that?" is not answerable by reading the code. It is
only answerable by reading a record of what happened — and if you did not write
it down as the run happened, **the information is gone**, because the model may
decide differently next time.

> Tracing is not a nice-to-have you bolt on when things get hard. An untraced
> agent in production is a system whose failures you can only apologise for.


In [ ]:
# A full trace. This is what you read when something goes wrong.
run = Agent().run("Is order ACME-1046 refundable? The reason is changed_mind.")
print(run.trace.show())

In [ ]:
# The trace also answers questions ABOUT the run, without you re-reading it.
t = run.trace
print("tool call sequence :", " → ".join(t.call_sequence()))
print("distinct tools used:", t.tools_used())
print("error rate         :", f"{t.error_rate():.0%}")
print("wall clock         :", f"{t.duration_s:.3f}s")
print("steps              :", len(t.steps))
print()
print("Context growth per step (this is your cost curve):")
for s in t.steps:
    print(f"  step {s.step}: ~{s.context_tokens:>5} tokens in, {s.duration_ms:>6.1f} ms")

`call_sequence()` is the trace reduced to its skeleton, and it is what you assert
on when testing an agent. You cannot assert "the answer is correct" cheaply; you
*can* assert "it called `get_order_status` before `check_refund_eligibility`".

**Testing an agent means testing trajectories, not outputs.**


In [ ]:
# A trace serialises. This is the bug-report format: everything a colleague
# needs to see what your agent did — no code, no key, no ability to reproduce
# a stochastic failure required.
from agent_core import Trace
import json

blob = run.trace.to_json()
print(blob[:520], "…\n")

replayed = Trace.from_dict(json.loads(blob))
print("replayed steps  :", len(replayed.steps))
print("replayed calls  :", " → ".join(replayed.call_sequence()))
print("identical?      :", replayed.call_sequence() == run.trace.call_sequence())

## WHAT — reflection, and when it is worth paying for

**Reflection** = the agent reviews its own draft answer against the evidence
before returning it.

It works — but not for the reason people usually give. It is not the model
"thinking harder". It works because **checking a claim against a list of facts
is a genuinely easier task than producing the claim was**, and because the second
pass sees the draft as text to audit rather than as its own in-progress
reasoning.

That also tells you its limits. Narrow, checkable criteria ("is this figure in
the evidence?") improve reliably. Open-ended ones ("is this good?") mostly do not.

> ### When NOT to reflect
> Reflection **doubles latency and cost on every answer**. It earns that on
> high-stakes or hallucination-prone outputs. It does not on a lookup that
> returned one unambiguous number. "Reflect on everything" is a common and
> expensive mistake — which is why our default is **off**.


In [ ]:
# Reflection compares a draft against the evidence actually gathered.
from agent_core.control import REFLECTION_PROMPT

plain     = Agent(use_reflection=False).run("What is the status of order ACME-1044?")
reflected = Agent(use_reflection=True ).run("What is the status of order ACME-1044?")

print(compare({"no reflection": plain.trace, "with reflection": reflected.trace}))
print()
print("The check it performs (grounding / completeness / honesty):")
print(REFLECTION_PROMPT.split("Check three things")[1][:340])
print()
print("NOTE: the offline mock cannot genuinely critique — it has no judgement,")
print("so it approves and says so. That is honest rather than a staged demo;")
print("real reflection needs a real model. Re-run this cell with a key set.")

## HOW (parallel mapping) — control and tracing in production

| We build | LangGraph / LangSmith |
|---|---|
| `budget_exceeded()` | `recursion_limit` |
| `repetition()`, `error_streak()`, `no_new_information()` | **you still write these** |
| `Trace` / `StepRecord` | LangSmith run trees |
| `trace.to_json()` as a bug report | a shareable LangSmith URL |
| `reflect()` | a critique node in the graph |

### Read the second row twice

`recursion_limit` is a **backstop**. It tells you the graph hit its ceiling. It
never tells you the agent called the same tool with the same arguments five
times.

So adopting a framework does **not** give you the thing this section argues is
the actual engineering. It gives you the safety net and leaves the diagnosis to
you — and because `recursion_limit` *looks* like it has the problem covered, the
diagnostic conditions are the thing teams most often never get round to writing.

The good news: everything in `agent_core/control.py` ports directly, because it
was never framework-specific. It is reasoning over the message history.
`agent_lc/graph.py` does exactly that port.


In [ ]:
# ============================================================
# LANGGRAPH TRACK — needs OPENAI_API_KEY
# ============================================================
# Everything above ran offline on the mock. From here we use a REAL model,
# because this half of the session is about what you actually deploy.
# Without a key these cells skip cleanly — the from-scratch cells above have
# already made the conceptual point.
import os

LC_READY = bool(os.getenv("OPENAI_API_KEY"))
if LC_READY:
    from langchain_openai import ChatOpenAI
    chat = ChatOpenAI(model=os.getenv("AGENT_LLM_MODEL", "gpt-4o-mini"), temperature=0)
    print("LangGraph track: ready ->", chat.model_name)
else:
    chat = None
    print("LangGraph track: SKIPPED (no OPENAI_API_KEY).")
    print("Set a key to run these cells. The from-scratch cells above still ran.")

In [ ]:
# The port, running offline on a test double so the comparison is reproducible.
# (agent_lc.fake_model exists for tests, not teaching — see its docstring.)
from langchain_core.messages import HumanMessage
from agent_lc.graph import build_agent_graph, build_diagnostic_graph
from agent_lc.fake_model import FakeToolCallingModel

goal = "What is the status of order ACME-1042?"
start = lambda: {"messages": [HumanMessage(goal)], "steps": 0, "stop_reason": None}

budget_only = build_agent_graph(FakeToolCallingModel(fault="loop_forever"), max_steps=8)
diagnostic  = build_diagnostic_graph(FakeToolCallingModel(fault="loop_forever"), max_steps=8)

b = budget_only.invoke(start())
d = diagnostic.invoke(start())

print(f"budget-only  steps={b['steps']}  stop={b['stop_reason']}")
print(f"diagnostic   steps={d['steps']}  stop={d['stop_reason']}")
print()
print("Same lesson as the from-scratch comparison, same numbers, different engine.")

### LangSmith — what a real tracing backend buys you

Turning it on is **two environment variables**. No code changes, no decorators:

```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "..."
```

Everything in `agent_lc` is then traced, because LangChain instruments its own
primitives. What you get over a printed string:

- **persistence and search** — *"every run last week where the agent escalated"*
  is a query, not a grep through notebook output
- **cost and latency per step**, aggregated across runs
- **diffing** two runs of the same task after a prompt change
- **datasets and evaluators** — `data/tasks/agent_tasks.jsonl` becomes a
  LangSmith dataset you run on every commit
- **a shareable URL** — the "trace as a bug report" argument, except your
  colleague clicks a link

This is a genuine reason to adopt the framework. Our `Trace` taught you what to
look at; LangSmith is where you look at it when there are ten thousand runs.


In [ ]:
from agent_lc import langsmith_status
print(langsmith_status())

## Recap

- Three different questions: **done**, **must stop**, **stuck**. The third is
  where the money goes.
- Diagnostic conditions before the budget backstop — same safety, far better
  traces, and a fraction of the spend.
- Stopping early still owes the user an honest partial answer.
- **The trace is the debugging interface.** Untraced agent failures are
  unreproducible and therefore unfixable.
- Assert on **trajectories**, not outputs.
- Reflection is a real technique with a real price. Use it deliberately.

**Next → Notebook 06 (Failure modes):** five broken agents. You diagnose them
from the trace.
